# PS 5 — Adult Census Income (UCI)

**DWM practical | Run cells top-to-bottom.**

All datasets are read from `../data/` and results saved in `../outputs/`. Run `python download_datasets.py` in the repo root beforehand.

**Goal:** inspect numeric/categorical/Boolean-like types, replace `?`, impute, clean, justify nominal vs ordinal encoding, and normalize numeric predictors. `income` is the target.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
DATA = Path('../data')
OUT = Path('../outputs'); OUT.mkdir(exist_ok=True)

cols = ['age','workclass','fnlwgt','education','education.num','marital.status','occupation',
        'relationship','race','sex','capital.gain','capital.loss','hours.per.week','native.country','income']
df = pd.read_csv(DATA/'adult.data',header=None,names=cols,skipinitialspace=True,na_values=['?',' ?'])
before = df.copy()

In [ ]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Dtypes:'); print(df.dtypes)
print('Missing values:'); print(df.isna().sum())
print('Duplicates:', df.duplicated().sum())
display(df.head())
display(df.describe(include='all').T)

**Column names matter:** this version uses `education.num` (a dot), not `education_num`. `education.num` already represents an ordered education level. `sex` and `income` are binary; other categorical predictors are nominal.

In [ ]:
cat_cols = df.select_dtypes(include=['object','string']).columns.tolist()
num_cols = df.select_dtypes(include='number').columns.tolist()
for c in cat_cols:
    df[c] = df[c].astype('string').str.strip().str.lower().replace({'?':pd.NA,'':pd.NA})
    df[c] = df[c].fillna(df[c].mode(dropna=True).iloc[0])
for c in num_cols:
    df[c] = pd.to_numeric(df[c],errors='coerce').fillna(df[c].median())
print('Numeric:',num_cols)
print('Categorical:',cat_cols)
print('Boolean-like:', ['sex','income'])
print('Nulls after cleaning:',df.isna().sum().sum())

**Encoding decisions:** income `<=50k`→0 and `>50k`→1 (target), sex `female`→0, `male`→1 (binary); education.num is ordered and kept numeric; education text is redundant with education.num, so drop it; workclass, marital.status, occupation, relationship, race, native.country receive one-hot encoding.

In [ ]:
df['income'] = df['income'].map({'<=50k':0,'>50k':1}).astype(int)
df['sex'] = df['sex'].map({'female':0,'male':1}).astype(int)
df = df.drop(columns=['education'])
nominal = ['workclass','marital.status','occupation','relationship','race','native.country']
scale_cols = ['age','fnlwgt','capital.gain','capital.loss','hours.per.week']
print('Before scaling:');display(df[scale_cols].head())
df[scale_cols] = MinMaxScaler().fit_transform(df[scale_cols])
print('After scaling:');display(df[scale_cols].head())
df = pd.get_dummies(df,columns=nominal,dtype=int)

In [ ]:
print('BEFORE preprocessing:'); display(before.head())
print('AFTER preprocessing:'); display(df.head())
print('Final shape:', df.shape)
print('Remaining missing values:', df.isna().sum().sum())
print('Remaining duplicates:', df.duplicated().sum())

In [ ]:
df.to_csv(OUT/'PS05_adult_processed.csv',index=False)